# D · 원하람 — Logistic Regression (D-1 ~ D-2d)

`haram` 브랜치. Accelerator None. 융합은 `D_fusion_kaggle.ipynb`.

## 1. 저장소·환경

In [ ]:
%cd /kaggle/working
!rm -rf LLM-name-tagging
!git clone -b haram https://github.com/seoultech-Machine-Learning-26-2/LLM-name-tagging.git
%cd /kaggle/working/LLM-name-tagging
!git log -1 --oneline

import os
assert os.path.exists('experiments/D_logistic_regression.py'), 'haram 브랜치가 push 됐는지 확인'

import platform, sys, numpy, pandas, sklearn, scipy
print(platform.platform(), '| Python', sys.version.split()[0])
print('numpy', numpy.__version__, '| pandas', pandas.__version__,
      '| sklearn', sklearn.__version__, '| scipy', scipy.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'GPU 없음'

## 1-1. 한글 폰트

In [ ]:
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1
!fc-cache -f > /dev/null 2>&1
!rm -rf ~/.cache/matplotlib
import matplotlib.font_manager as fm
print('NanumGothic:', any('Nanum' in f.name for f in fm.fontManager.ttflist))

## 2. Logistic Regression (D-1 ~ D-2d)

C 격자 · TF-IDF 차원 · 질문 중심화. CPU 약 5분.

In [ ]:
!python -u -W ignore -m experiments.D_logistic_regression

## 3. 결과

In [ ]:
import pandas as pd
from IPython.display import Image, display as d
pd.set_option('display.width', 220)

res = pd.read_csv('results/course/D/logistic_regression/val_results.csv')
cols = ['run_id','feature_set','train_model_BA','model_BA','accuracy','macro_f1',
        'log_loss','ece','sibling_acc','company_BA','converged','n_iter','fit_seconds']
display(res[[c for c in cols if c in res.columns]].round(4))

### 3-1. train/val 차이

In [ ]:
lr = res[res.model == 'LogisticRegression'].copy()
lr['gap'] = lr.train_model_BA - lr.model_BA
display(lr[['config','train_model_BA','model_BA','gap']].round(4).sort_values('gap', ascending=False))
d(Image('results/course/D/logistic_regression/fig_config_comparison.png'))

### 3-2. 혼동행렬

In [ ]:
d(Image('results/course/D/logistic_regression/fig_confusion_val.png'))

### 3-3. 오분류 5개

In [ ]:
err = pd.read_csv('results/course/D/logistic_regression/val_errors_seed42.csv')
for _, r in err.iterrows():
    print(f"[{r.id}] {r.true_model} -> {r.pred_model} "
          f"(p={r.pred_prob:.3f}, 정답 p={r.true_prob:.3f}, 같은 개발사={r.same_company})")
    print('   ', r.text_head[:200], '\n')

### 3-4. 계수

In [ ]:
coef = pd.read_csv('results/course/D/logistic_regression/coefficients_style.csv')
display(coef.feature.value_counts().head(12).to_frame('상위 등장 모델 수'))
display(coef.sort_values('abs_coef', ascending=False).groupby('model').head(1)
            [['model','feature','coef']].sort_values('model').reset_index(drop=True))

### 3-5. 보정·형제 혼동·모델별 재현율

In [ ]:
B = 'results/course/D/logistic_regression/'
print('확신도 구간별 (gap > 0 이면 과신)')
display(pd.read_csv(B+'val_calibration.csv'))
print('오분류 중 같은 개발사 비율')
display(pd.read_csv(B+'val_sibling_errors.csv'))
print('모델별 재현율')
display(pd.read_csv(B+'val_per_class.csv'))

### 3-6. 질문 중심화 효과 (D-2d)

In [ ]:
lr = res[res.model == 'LogisticRegression']
piv = (lr.pivot_table(index=['feature_set','max_features'], columns='centering',
                      values='model_BA', aggfunc='max')
         .reindex(columns=['raw','centered','concat']))
display(piv.round(4))

### 3-7. 신뢰구간과 쌍별 검정

In [ ]:
ci = pd.read_csv(B+'val_ci.csv')
m = lr[['config','model_BA']].drop_duplicates('config')
display(ci.merge(m, on='config').sort_values('model_BA', ascending=False).head(8).round(4))

pw = pd.read_csv(B+'val_pairwise.csv')
print(f"최고 설정이 통계적으로 앞선 설정: {int(pw.best_is_better.sum())}/{len(pw)}")
display(pw[~pw.best_is_better].round(4))

### 3-8. 개발사별 분해

In [ ]:
display(pd.read_csv(B+'val_company_breakdown.csv').round(4))

## 4. 다음 단계
융합(D-3 ~ D-4)은 `notebooks/D_fusion_kaggle.ipynb`.

## 5. 결과 저장

In [ ]:
!zip -qr /kaggle/working/D_results.zip results/course/D preds/val_D_* preds/test_D_* 2>/dev/null
!ls -lh /kaggle/working/D_results.zip
!unzip -l /kaggle/working/D_results.zip | tail -25

## 6. test 1회 (팀 합의 후에만)

In [ ]:
# !python -u -W ignore -m experiments.D_logistic_regression --final combined_C10.0